In [1]:
train_data= "../dataset/preprocessed/training"
test_data= "../dataset/preprocessed/testing"

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.15.0+cu134
True


In [3]:
from torch.utils.data import Dataset
import os
from PIL import Image

class BTDataset(Dataset):
    def __init__(self, root, transform= None):
        self.root= root
        self.transform = transform
        self.samples= []
        self.classes= sorted(os.listdir(root))

        for label, class_name in enumerate(self.classes):
            class_dir= os.path.join(root, class_name)

            if not os.path.isdir (class_dir):
                continue
            for image in os.listdir(class_dir):
                self.samples.append((os.path.join(class_dir, image), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label= self.samples[idx]
        img= Image.open(img_path).convert("L").resize((256, 256))
        if self.transform:
            img= self.transform(img)
        return img, label

In [4]:
from torchvision import transforms

train_transform= transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean= [0.184], std= [0.201])
    # transforms.Normalize(
    #     mean= [0.18395672, 0.18395881, 0.18399635],
    #     std= [0.20059544, 0.20059426, 0.20062249])
])

val_transform= transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean= [0.184], std= [0.201])
    # transforms.Normalize(
    #     mean= [0.18395672, 0.18395881, 0.18399635],
    #     std= [0.20059544, 0.20059426, 0.20062249])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean= [0.184], std= [0.201])
    # transforms.Normalize(
    #     mean=[0.18395672, 0.18395881, 0.18399635],
    #     std=[0.20059544, 0.20059426, 0.20062249])
])

In [5]:
train_dataset= BTDataset(train_data, transform= train_transform)
val_dataset= BTDataset(train_data, transform= val_transform)
test_dataset = BTDataset(test_data, transform=test_transform)

In [6]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

5600
5600
1600


In [7]:
from torch.utils.data import random_split

total_size= len(train_dataset)
train_size = int(total_size * 0.9)
val_size= total_size- train_size

generator= torch.Generator().manual_seed(42)

train_split, val_split= random_split(
    train_dataset,
    [train_size, val_size],
    generator= generator
)

In [8]:
from torch.utils.data import Subset

train_subset= Subset(
    train_dataset,
    train_split.indices
)

val_subset= Subset(
    val_dataset,
    val_split.indices
)

In [10]:
from torch.utils.data import DataLoader

train_loader= DataLoader(
    dataset= train_subset,
    batch_size= 64,
    shuffle= True,
    pin_memory= True
)

val_loader= DataLoader(
    dataset= val_subset,
    batch_size= 64,
    shuffle= False,
    pin_memory= True
)

test_loader= DataLoader(
    dataset= test_dataset,
    batch_size= 64,
    shuffle= False,
    pin_memory= True
)

In [13]:
print(len(train_split))
print(len(val_split))
print(len(test_dataset))

4480
1120
1600


In [14]:
import torch.nn as nn

def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(),
        nn.Conv2d(cout, cout, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )

class BTCCnn(nn.Module):
    def __init__(self, num_classes=4, in_ch=1):
        super(BTCCnn, self).__init__()

        self.features = nn.Sequential(
            conv_block(in_ch, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
            conv_block(256, 512),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model = BTCCnn(num_classes=4, in_ch=1)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

Parameters: 4,878,372


In [16]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model= model.to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer= torch.optim.Adam(
    model.parameters(), lr= 0.001
)